#### Задача 1

Сгенерируйте DataFrame на 5 миллионов строк с колонками id (от 0 до 5 000 000), age (от 18 до 99), salary (float от 30 000.0 до 500 000.0) и is_active (0 или 1).

Измерьте исходный объем памяти через .memory_usage(deep=True).sum() и уменьшите его минимум в 3 раза, подобрав минимально достаточные типы данных. Проверьте, не потерялась ли точность в колонке salary.

#### Задача 2

Дан DataFrame с колонками age (int), city (object, 50 значений), score (float). 300 000 строк. Отфильтруйте строки цепочкой:

    tmp = df[df['age'] > 30]
    tmp = tmp[tmp['city'] == 'Moscow']
    tmp = tmp[tmp['score'] > 0.5]

Замерьте время.

Затем сделайте то же одним loc с объединённым условием:

    result = df.loc[(df['age'] > 30) & (df['city'] == 'Moscow') & (df['score'] > 0.5)]

Снова замерьте время. Проверьте, что результаты совпадают.

**Усложнение:** приведите city к category и повторите оба варианта. Сравните время.

#### Задача 3

Дан DataFrame на 100 000 строк с колонками:

- id (int64)

- name (object, 100 000 уникальных строк)

- description (object, длинные строки, ~200 символов)

- price (float64)

- is_active (bool)

Задача:

1. Выведите df.memory_usage(deep=True) — по каждой колонке.

2. Найдите колонку, которая занимает больше всего памяти. Объясните в комментариях, почему так.

3. Посчитайте, какую долю от общей памяти занимает description. Предложите способ её сократить.

#### Задача 4

Сгенерируйте DataFrame на 10 миллионов строк со смешанными типами данных (числа, даты, строки) и сохраните его в CSV. Теперь представьте, что оперативной памяти у вас хватает только на 1 миллион строк. Напишите скрипт, который читает получившийся CSV через параметр chunksize, фильтрует данные и считает агрегированную статистику без переполнения памяти.


#### Задача 5

Даны две таблицы:

- orders: 1 000 000 строк, колонки order_id, customer_id, amount.

- customers: 50 000 строк, колонки customer_id, segment (object, 5 уникальных значений).

Нужно добавить в orders колонку segment по customer_id.

Задача:

1. Сделайте это через orders.merge(customers, on='customer_id', how='left'). Замерьте время и итоговое число строк (проверьте, не размножились ли строки).

2. Сделайте то же через orders['customer_id'].map(customers.set_index('customer_id')['segment']). Замерьте время.

3. Сравните результаты (должны совпадать) и время. Объясните, в каком случае map предпочтительнее, а в каком без merge не обойтись (**подсказка:** map работает только для соответствия «один ключ → одно значение»).

Условие по памяти: перед merge приведите customers['segment'] к category и сравните пиковое потребление памяти результата merge до и после.

#### Задача 6 (*) - работа с transform.

Дан DataFrame с колонками team (категориальная, 50 команд), player (object), score (float) на 1 млн строк.

Задача:

- Через groupby('team').apply(...) добавьте колонку score_pct — долю очков игрока от суммы очков его команды. Замерьте время.

- Сделайте то же через transform('sum') и деление. Замерьте время.

- Проверьте, что результаты совпадают.

- Объясните, почему transform предпочтительнее, когда результат должен иметь ту же длину, что и исходный DataFrame.

transform reference: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html

#### Задача 7 (**) - ускорение пайплайна.

Дан «медленный» код. Нужно найти узкие места и ускорить его, не меняя логику. Можете сами сгенерировать синтетические данные для проверки, но с фиксированным seed, чтобы результаты были воспроизводимы.


In [1]:
def slow_pipeline(df):
    # Шаг 1: фильтрация
    filtered = df[df['status'] == 'active'].copy()

    # Шаг 2: группировка и применение кастомной функции
    def calc_metric(group):
        group['ratio'] = group['value'] / group['baseline']
        group['flag'] = group['ratio'].apply(lambda x: 1 if x > 1.5 else 0)
        return group

    result = filtered.groupby('category').apply(calc_metric)

    # Шаг 3: объединение с другой таблицей
    other = pd.read_csv('other.csv')  # 500k строк, ключ 'id'
    merged = result.merge(other, on='id', how='left')

    return merged